# Distortion additivity experiment

This notebook tests whether output-space distortion is additive across pruned layers.

For each pruning ratio, it compares:

- the measured distortion from pruning one layer at a time;
- the measured distortion from pruning multiple layers jointly;
- the sum of the single-layer distortions.

Additivity would mean that the joint distortion is close to the sum of the individual distortions. The notebook reports both mean per-example MSE and worst-case per-example MSE. It uses the shared dense checkpoint and does not fine-tune the masked models, so the measurement isolates the distortion model itself.

## Before running

1. Use a GPU runtime in Colab.
2. Add a Colab Secret named GITHUB_TOKEN with read access to the private my-thesis repository and its private submodules.
3. Mount Google Drive when prompted.
4. Ensure the dense checkpoint exists at /content/drive/MyDrive/rdhp_rdp_runs_seed_42/dense_resnet32.pt.
5. Ensure CIFAR-10 is extracted at /content/drive/MyDrive/cifar10/cifar-10-batches-py.

The experiment is deliberately smaller than the training run: it reuses the existing checkpoint and evaluates a calibration subset. Change the configuration cell to increase the number of layers, pairs, groups, or calibration examples.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys

OWNER = 'hamidrg20001379'
REPO_NAME = 'my-thesis'
REPO_URL = f'https://github.com/{OWNER}/{REPO_NAME}.git'
REPO = Path('/content') / REPO_NAME

if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout. Remove it and rerun this cell.')

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with private-repository read access: ')
if not token:
    raise RuntimeError('A GitHub token is required.')

askpass = Path('/tmp/github-askpass.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\n' x-access-token ;;\n"
    "  *Password*) printf '%s\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})
try:
    if not REPO.exists():
        subprocess.run(
            ['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO)],
            env=git_env, check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'],
            env=git_env, check=True,
        )
finally:
    askpass.unlink(missing_ok=True)

print('Repository:', REPO)

In [ ]:
import copy
import json
import random
import time
from argparse import Namespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from google.colab import drive
from rdp_resnet32_adapter import (
    load_resnet32,
    logits,
    make_loaders,
    mask_topk,
    prunable_layers,
)

SEED = 42
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 256
TEST_SAMPLES = 256
BATCH_SIZE = 128

EXPERIMENTS = REPO / 'working' / 'experiments'
sys.path.insert(0, str(EXPERIMENTS))

# Use a small layer subset by default. Increase these for a more exhaustive test.
LAYER_COUNT = 12
PAIR_COUNT = 24
GROUP_COUNT = 12
GROUP_SIZE = 4
RATIOS = [0.25, 0.50, 0.75]

DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.exists():
    drive.mount('/content/drive')

DATA_ROOT = DRIVE_ROOT / 'cifar10'
RUN_ROOT = DRIVE_ROOT / f'rdhp_rdp_runs_seed_{SEED}'
CHECKPOINT = RUN_ROOT / 'dense_resnet32.pt'
OUTPUT_ROOT = RUN_ROOT / 'distortion_additivity'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing dense checkpoint: {CHECKPOINT}')

dataset_dir = DATA_ROOT / 'cifar-10-batches-py'
required = [dataset_dir / name for name in [
    'data_batch_1', 'data_batch_2', 'data_batch_3',
    'data_batch_4', 'data_batch_5', 'test_batch',
]]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Incomplete CIFAR-10 dataset: ' + ', '.join(missing))

print('Output directory:', OUTPUT_ROOT)

## Load the common dense model

The dense checkpoint is loaded once. All masked models start from this identical state. The calibration and test loaders use deterministic subsets from the existing adapter.

In [ ]:
from rdp_resnet32_adapter import seed_everything

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loader_args = Namespace(
    synthetic=False,
    data_root=str(DATA_ROOT),
    download=False,
    train_samples=TRAIN_SAMPLES,
    calib_samples=CALIB_SAMPLES,
    test_samples=TEST_SAMPLES,
    batch_size=BATCH_SIZE,
    seed=SEED,
)
(train_loader, calibration_loader, test_loader), split_info = make_loaders(loader_args)

payload = torch.load(CHECKPOINT, map_location=device, weights_only=False)
state_dict = payload['model_state_dict'] if 'model_state_dict' in payload else payload
model = load_resnet32().to(device)
model.load_state_dict(state_dict)
model.eval()

layers = prunable_layers(model)
layer_names = [
    name for name, module in model.named_modules()
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
]
if len(layer_names) != len(layers):
    raise RuntimeError('Layer-name and layer-list mismatch.')

selected_layers = np.linspace(
    0, len(layers) - 1, min(LAYER_COUNT, len(layers)), dtype=int
).tolist()
selected_layers = list(dict.fromkeys(int(i) for i in selected_layers))

dense_calib = logits(model, calibration_loader, device)
dense_test = logits(model, test_loader, device)
print('Device:', device)
print('Prunable layers:', len(layers))
print('Selected layers:', [(i, layer_names[i]) for i in selected_layers])
print('Calibration logits:', tuple(dense_calib.shape))
print('Test logits:', tuple(dense_test.shape))

In [ ]:
def all_one_masks():
    return [
        torch.ones_like(layer.weight, dtype=torch.bool)
        for layer in layers
    ]

def masks_for(indices_and_ratios):
    masks = all_one_masks()
    for layer_index, ratio in indices_and_ratios:
        masks[layer_index] = mask_topk(
            layers[layer_index].weight, float(ratio)
        ).detach().cpu()
    return masks

def masked_copy(masks):
    masked = copy.deepcopy(model).to(device)
    with torch.no_grad():
        for layer, mask in zip(prunable_layers(masked), masks):
            layer.weight.mul_(mask.to(device=device, dtype=layer.weight.dtype))
    masked.eval()
    return masked

@torch.no_grad()
def measure_masks(masks):
    masked = masked_copy(masks)
    masked_calib = logits(masked, calibration_loader, device)
    masked_test = logits(masked, test_loader, device)

    calib_error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
    test_error = (masked_test - dense_test).flatten(1).square().mean(dim=1)
    return {
        'calib_mean': float(calib_error.mean()),
        'calib_max': float(calib_error.max()),
        'test_mean': float(test_error.mean()),
        'test_max': float(test_error.max()),
    }

single_lookup = {}
single_rows = []
started = time.monotonic()

for layer_index in selected_layers:
    for ratio in RATIOS:
        values = measure_masks(masks_for([(layer_index, ratio)]))
        row = {
            'layer_index': layer_index,
            'layer_name': layer_names[layer_index],
            'ratio': ratio,
            **values,
        }
        single_rows.append(row)
        single_lookup[(layer_index, ratio)] = row

single_df = pd.DataFrame(single_rows)
print(f'Single-layer measurements: {len(single_df)}')
print(f'Elapsed seconds: {time.monotonic() - started:.1f}')
display(single_df.head())

## Joint pruning and additivity residuals

For a pair or group of layers, define:

- D_joint: measured distortion after pruning all selected layers jointly;
- D_sum: sum of the corresponding single-layer distortions;
- interaction = D_joint - D_sum;
- ratio = D_joint / D_sum.

A ratio near 1 and an interaction near 0 support additivity. Large positive or negative interactions indicate cross-layer effects.

In [ ]:
rng = np.random.default_rng(SEED)
pair_candidates = [
    (int(a), int(b))
    for position, a in enumerate(selected_layers)
    for b in selected_layers[position + 1:]
]
if not pair_candidates:
    raise RuntimeError('Select at least two layers for pair testing.')
pair_count = min(PAIR_COUNT, len(pair_candidates))
pair_indices = [
    pair_candidates[i]
    for i in rng.choice(len(pair_candidates), size=pair_count, replace=False)
]

pair_rows = []
for layer_a, layer_b in pair_indices:
    for ratio in RATIOS:
        values = measure_masks(
            masks_for([(layer_a, ratio), (layer_b, ratio)])
        )
        a = single_lookup[(layer_a, ratio)]
        b = single_lookup[(layer_b, ratio)]
        for metric in ('calib_mean', 'calib_max', 'test_mean', 'test_max'):
            joint = values[metric]
            additive = a[metric] + b[metric]
            values[f'{metric}_sum'] = additive
            values[f'{metric}_interaction'] = joint - additive
            values[f'{metric}_ratio'] = joint / max(additive, 1e-12)
        pair_rows.append({
            'layer_a': layer_a,
            'layer_a_name': layer_names[layer_a],
            'layer_b': layer_b,
            'layer_b_name': layer_names[layer_b],
            'ratio': ratio,
            **values,
        })

pair_df = pd.DataFrame(pair_rows)

group_rows = []
group_ratio = 0.50
for group_id in range(min(GROUP_COUNT, 100)):
    group = rng.choice(
        selected_layers, size=min(GROUP_SIZE, len(selected_layers)),
        replace=False,
    ).tolist()
    values = measure_masks(
        masks_for([(int(i), group_ratio) for i in group])
    )
    row = {
        'group_id': group_id,
        'layer_indices': ','.join(str(int(i)) for i in group),
        'layer_names': ' | '.join(layer_names[int(i)] for i in group),
        'ratio': group_ratio,
        **values,
    }
    for metric in ('calib_mean', 'calib_max', 'test_mean', 'test_max'):
        additive = sum(
            single_lookup[(int(i), group_ratio)][metric] for i in group
        )
        row[f'{metric}_sum'] = additive
        row[f'{metric}_interaction'] = values[metric] - additive
        row[f'{metric}_ratio'] = values[metric] / max(additive, 1e-12)
    group_rows.append(row)

group_df = pd.DataFrame(group_rows)
print('Pair measurements:', len(pair_df))
print('Group measurements:', len(group_df))
display(pair_df.head())
display(group_df.head())

In [ ]:
def summarize(df, prefix):
    return {
        f'{prefix}_rows': int(len(df)),
        f'{prefix}_calib_mean_ratio_median': float(
            df['calib_mean_ratio'].median()
        ),
        f'{prefix}_calib_mean_relative_interaction_median': float(
            (df['calib_mean_interaction'] /
             df['calib_mean_sum'].clip(lower=1e-12)).median()
        ),
        f'{prefix}_test_mean_ratio_median': float(
            df['test_mean_ratio'].median()
        ),
        f'{prefix}_test_mean_relative_interaction_median': float(
            (df['test_mean_interaction'] /
             df['test_mean_sum'].clip(lower=1e-12)).median()
        ),
        f'{prefix}_calib_max_ratio_median': float(
            df['calib_max_ratio'].median()
        ),
        f'{prefix}_test_max_ratio_median': float(
            df['test_max_ratio'].median()
        ),
    }

summary = {
    'seed': SEED,
    'device': str(device),
    'checkpoint': str(CHECKPOINT),
    'calibration_samples': CALIB_SAMPLES,
    'test_samples': TEST_SAMPLES,
    'selected_layers': [
        {'index': int(i), 'name': layer_names[i]}
        for i in selected_layers
    ],
    'ratios': RATIOS,
    'definition': {
        'distortion': 'mean squared logit error versus dense model',
        'additive_prediction': 'sum of single-layer distortions',
        'interaction': 'joint distortion minus additive prediction',
    },
    **summarize(pair_df, 'pair'),
    **summarize(group_df, 'group'),
}

single_df.to_csv(OUTPUT_ROOT / 'single_layer_distortions.csv', index=False)
pair_df.to_csv(OUTPUT_ROOT / 'pair_additivity.csv', index=False)
group_df.to_csv(OUTPUT_ROOT / 'group_additivity.csv', index=False)
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].axhline(1.0, color='black', linewidth=1)
axes[0].scatter(
    pair_df['calib_mean_sum'], pair_df['calib_mean'],
    alpha=0.75, label='pairs',
)
axes[0].set_xlabel('Sum of single-layer distortions')
axes[0].set_ylabel('Joint distortion')
axes[0].set_title('Pair additivity: calibration mean MSE')
axes[0].legend()

axes[1].axhline(1.0, color='black', linewidth=1)
axes[1].scatter(
    group_df['calib_mean_sum'], group_df['calib_mean'],
    alpha=0.75, color='tab:orange', label='groups',
)
axes[1].set_xlabel('Sum of single-layer distortions')
axes[1].set_ylabel('Joint distortion')
axes[1].set_title('Group additivity: calibration mean MSE')
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_ROOT / 'additivity_scatter.png', dpi=160)
plt.show()

print(json.dumps(summary, indent=2))
print('Saved artifacts to:', OUTPUT_ROOT)

## Interpretation

Use the median ratios and residuals in summary.json:

- ratio near 1.0 means the measured joint distortion agrees with the additive prediction.
- Positive interaction means joint pruning is more damaging than the sum predicts.
- Negative interaction means the single-layer effects overlap or partially cancel.
- Mean-MSE additivity is the more natural comparison for a quadratic surrogate. Worst-case MSE uses a maximum over examples and is not expected to be additive in general.
- This is an output-space diagnostic of the reconstructed implementation. It does not by itself validate the paper's author code or establish hardware speedup.

## Optional: publish additivity results

After the experiment finishes, create a Colab Secret named GITHUB_WRITE_TOKEN with Contents read/write access to rdhp-rdp-colab-experiment. Then set PUBLISH_RESULTS = True in the next cell and run it.

The cell uploads only the additivity artifacts from the current RUN_ROOT.

In [ ]:
PUBLISH_RESULTS = False

if not PUBLISH_RESULTS:
    print('Publishing disabled. Set PUBLISH_RESULTS = True after the experiment finishes.')
else:
    from google.colab import userdata
    from pathlib import Path
    import os
    import shutil
    import subprocess

    token = userdata.get('GITHUB_WRITE_TOKEN')
    if not token:
        raise RuntimeError(
            'Add GITHUB_WRITE_TOKEN with Contents read/write access.'
        )

    source = OUTPUT_ROOT
    if not source.exists():
        raise FileNotFoundError(f'Additivity results not found: {source}')

    repo = Path('/content/rdhp-rdp-colab-additivity-upload')
    if repo.exists():
        shutil.rmtree(repo)

    askpass = Path('/tmp/github-askpass-additivity.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "  *Username*) printf '%s\n' x-access-token ;;\n"
        "  *Password*) printf '%s\n' \"$GITHUB_WRITE_TOKEN\" ;;\n"
        "  *) printf '\n' ;;\n"
        'esac\n'
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env.update({
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
        'GITHUB_WRITE_TOKEN': token,
    })

    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/hamidrg20001379/rdhp-rdp-colab-experiment.git',
            str(repo),
        ], env=env, check=True)

        destination = (
            repo / 'results' / RUN_ROOT.name / 'distortion_additivity'
        )
        shutil.copytree(source, destination, dirs_exist_ok=True)

        subprocess.run([
            'git', '-C', str(repo), 'config',
            'user.name', 'Colab Results Bot',
        ], check=True)
        subprocess.run([
            'git', '-C', str(repo), 'config',
            'user.email', 'colab-results@users.noreply.github.com',
        ], check=True)
        subprocess.run([
            'git', '-C', str(repo), 'add', 'results',
        ], check=True)

        changed = subprocess.run([
            'git', '-C', str(repo), 'diff', '--cached', '--quiet',
        ]).returncode != 0

        if changed:
            subprocess.run([
                'git', '-C', str(repo), 'commit', '-m',
                'Publish distortion additivity results',
            ], check=True)
            subprocess.run([
                'git', '-C', str(repo), 'push', 'origin', 'HEAD:master',
            ], env=env, check=True)
            print('Additivity results uploaded successfully.')
        else:
            print('No new additivity results to upload.')
    finally:
        askpass.unlink(missing_ok=True)